# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [ ]:
# votre réponse ici
def transposee(A):
    """Transposée de la matrice A, donnée comme liste de lignes."""
    lignes = len(A)
    colonnes = len(A[0])
    return [[A[i][j] for i in range(lignes)] for j in range(colonnes)]

print(transposee([[1, 2, 3], [4, 5, 6]]))


## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [ ]:
# votre réponse ici
def produit(A, B):
    """Produit matriciel A x B, en Python pur."""
    n, p = len(A), len(A[0])
    if len(B) != p:
        raise ValueError("dimensions incompatibles")
    q = len(B[0])
    C = [[0] * q for _ in range(n)]
    for i in range(n):
        for j in range(q):
            somme = 0
            for k in range(p):
                somme = somme + A[i][k] * B[k][j]
            C[i][j] = somme
    return C

identite = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
print(produit(A, identite) == A)
print(produit([[1, 2], [3, 4]], [[0, 1], [1, 0]]))

## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [ ]:
# votre réponse ici
import random
import time

random.seed(42)

def matrice_aleatoire(dimension):
    return [[random.random() for _ in range(dimension)] for _ in range(dimension)]

for taille in (40, 80, 120):
    mat_x = matrice_aleatoire(taille)
    mat_y = matrice_aleatoire(taille)
    t_depart = time.perf_counter()
    produit(mat_x, mat_y)
    print(f"n = {taille:>3} : {time.perf_counter() - t_depart:.4f} s")

## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [ ]:
# votre réponse ici
n = 120
A = matrice_aleatoire(n)
B = matrice_aleatoire(n)

debut = time.perf_counter()
produit(A, B)
t_python = time.perf_counter() - debut

An, Bn = np.array(A), np.array(B)
debut = time.perf_counter()
An @ Bn
t_numpy = time.perf_counter() - debut

print(f"python pur {t_python:.4f} s")
print(f"numpy      {t_numpy:.6f} s")
print("rapport :", round(t_python / max(t_numpy, 1e-9)))

## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [ ]:
# votre réponse ici
def resoudre(A, b, tolerance=1e-14):
    """Résout A x = b par le pivot de Gauss avec pivot partiel.

    A est une liste de lignes, b une liste. Ni A ni b ne sont modifiés.
    Un pivot inférieur ou égal à tolerance est considéré comme nul.
    """
    n = len(A)
    
    M = [list(A[i]) + [b[i]] for i in range(n)]

    for k in range(n):
        
        meilleure = max(range(k, n), key=lambda i: abs(M[i][k]))
        if abs(M[meilleure][k]) <= tolerance:
            raise ValueError("systeme singulier ou mal pose")
        M[k], M[meilleure] = M[meilleure], M[k]

        
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] = M[i][j] - facteur * M[k][j]

    
   


## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [ ]:
# votre réponse ici
random.seed(7)
dim_n = 50
mat_a = matrice_aleatoire(dim_n)
vec_b = [random.random() for _ in range(dim_n)]

res_perso = resoudre(mat_a, vec_b)
res_np = np.linalg.solve(np.array(mat_a), np.array(vec_b))

max_diff = max(abs(res_perso[idx] - res_np[idx]) for idx in range(dim_n))
print("ecart maximal :", max_diff)
print(max_diff < 1e-9)

## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


# votre réponse ici
Les chiffres parlent d'eux-mêmes. Toutes les composantes de la vraie solution valent 1. solve se trompe d'environ 0,7, et inv suivi d'un produit d'environ 21. La seconde méthode est donc à peu près trente fois pire que la première, et aucune des deux ne donne un résultat utilisable.
La cause est le conditionnement, ici de l'ordre de . Ce nombre mesure de combien une petite perturbation des données peut perturber la solution. Quand il approche , soit l'inverse de la précision des flottants, plus aucun chiffre du résultat n'est fiable, et aucun algorithme ne peut y remédier : c'est le problème lui-même qui est mal posé, pas votre programme. Refaites l'essai avec n = 10 : le conditionnement tombe à , les erreurs à  et , et l'écart de trente entre les deux méthodes, lui, demeure.
Retenez la règle pratique : pour résoudre un système, on écrit toujours np.linalg.solve(A, b), jamais np.linalg.inv(A) @ b.


## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [ ]:
# votre réponse ici
gauche = rotation(np.pi / 3) @ rotation(np.pi / 6)
droite = rotation(np.pi / 2)
print(np.allclose(gauche, droite))
print(np.round(droite, 10))

## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [ ]:
# votre réponse ici
def determinant(A):
    """Déterminant de A par élimination de Gauss avec pivot partiel."""
    n = len(A)
    M = [list(ligne) for ligne in A]
    det = 1.0
    for k in range(n):
        meilleure = max(range(k, n), key=lambda i: abs(M[i][k]))
        if abs(M[meilleure][k]) < 1e-14:
            return 0.0
        if meilleure != k:
            M[k], M[meilleure] = M[meilleure], M[k]
            det = -det
        det = det * M[k][k]
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] = M[i][j] - facteur * M[k][j]
    return det

A = [[1, 2, 2], [1, 3, -2], [3, 5, 8]]
print(determinant(A), float(np.linalg.det(np.array(A, dtype=float))))
print(determinant([[1, 2], [2, 4]]))

## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [ ]:
# votre réponse ici
from fractions import Fraction

def hilbert_exacte(n):
    return [[Fraction(1, i + j + 1) for j in range(n)] for i in range(n)]

n = 12
H_exacte = hilbert_exacte(n)
b_exact = [sum(ligne) for ligne in H_exacte]

try:
    resoudre(H_exacte, b_exact)
except ValueError as erreur:
    print("premier essai :", erreur)

x_exact = resoudre(H_exacte, b_exact, tolerance=0)
print(x_exact)
print(all(v == 1 for v in x_exact))

## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [ ]:
# votre réponse ici
def creer_translation(delta_x, delta_y):
    return np.array([[1.0, 0.0, delta_x],
                     [0.0, 1.0, delta_y],
                     [0.0, 0.0, 1.0]])

def creer_rotation_h(val_angle):
    c_val, s_val = np.cos(val_angle), np.sin(val_angle)
    return np.array([[c_val, -s_val, 0.0],
                     [s_val, c_val, 0.0],
                     [0.0, 0.0, 1.0]])

Pt = np.array([1.0, 0.0, 1.0])
Tr = creer_translation(3, 1)
Rot = creer_rotation_h(np.pi / 2)

print(np.round(Rot @ Tr @ Pt, 10))
print(np.round(Tr @ Rot @ Pt, 10))